# **pytorch 데이터다루기**

# 1.텐서 다루기 기초

In [1]:
import torch
import numpy as np
import pandas as pd

## (1) 텐서 만들기 : torch.tensor()


* 리스트, 넘파이로 부터

In [2]:
# 리스트
data = [[1, 2],[3, 4],[5, 6]]
x_data = torch.tensor(data)
x_data

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [3]:
# 넘파이 어레이
np_array = np.array(data)
x_np = torch.tensor(np_array)
x_np

tensor([[1, 2],
        [3, 4],
        [5, 6]])

* 데이터프레임은 지원하지 않음.

In [4]:
df = pd.DataFrame(data)
display(df)

x_df = torch.tensor(df)  # 오류 발생
x_df

,0,1
0,1,2
1,3,4
2,5,6


KeyError: 2

------------------------------------------------------------------------


## (2) 텐서의 속성


* 텐서의 속성 확인하기

In [5]:
print(x_np)
print(f"Shape : {x_np.shape}")
print(f"Datatype : {x_np.dtype}")
print(f"Device : {x_np.device}")

tensor([[1, 2],
        [3, 4],
        [5, 6]])
Shape : torch.Size([3, 2])
Datatype : torch.int64
Device : cpu


* 생성 시 데이터타입 지정
    * 보통, 다양한 연산을 위해서 dtype = torch.float32 로 지정

In [6]:
x_np = torch.tensor(np_array, dtype = torch.float32)

print(x_np)
print(f"Shape : {x_np.shape}")
print(f"Datatype : {x_np.dtype}")
print(f"Device : {x_np.device}")

tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
Shape : torch.Size([3, 2])
Datatype : torch.float32
Device : cpu


------------------------------------------------------------------------


## (3) Operations


* 텐서 작업을 디바이스(device)에 할당
    * cpu, gpu

In [7]:
# GPU가 사용 가능한 경우에는 GPU에, 그렇지 않은 경우에는 CPU에 할당
if torch.cuda.is_available():
    tensor = x_np.to("cuda")
else:
    tensor = x_np.to("cpu")

# 텐서가 어느 디바이스에 할당되어 있는지 확인
print(tensor.device)

cpu


* 넘파이처럼 다루기


In [8]:
np_array = np.array(data)
x_np = torch.tensor(np_array)
print(x_np)
print('='*30)

# 열, 행 인덱스로 조회
print(f"첫 행: {x_np[0]}")
print(f"첫 열: {x_np[:, 0]}")
print(f"마지막 열: {x_np[:, -1]}")
print('='*30)

# 값 수정
x_np[:,1] = 0
print(x_np)

tensor([[1, 2],
        [3, 4],
        [5, 6]])
첫 행: tensor([1, 2])
첫 열: tensor([1, 3, 5])
마지막 열: tensor([2, 4, 6])
tensor([[1, 0],
        [3, 0],
        [5, 0]])


* 결합 : torch.cat

In [9]:
np_array = np.array(data)
x_np = torch.tensor(np_array, dtype = torch.float32)
x_np

tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])

In [10]:
t0 = torch.cat([x_np, x_np], dim=0)
print(t0)

tensor([[1., 2.],
        [3., 4.],
        [5., 6.],
        [1., 2.],
        [3., 4.],
        [5., 6.]])


In [11]:
t1 = torch.cat([x_np, x_np], dim=1)
print(t1)

tensor([[1., 2., 1., 2.],
        [3., 4., 3., 4.],
        [5., 6., 5., 6.]])


## (4) 연산


* 집계함수

In [12]:
print(x_np)
print(x_np.sum())
print(x_np.mean())
print(x_np.std())

tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
tensor(21.)
tensor(3.5000)
tensor(1.8708)


* 행렬 곱

In [13]:
y1 = x_np @ x_np.T
y2 = x_np.matmul(x_np.T)

print(y1)
print(y2)

tensor([[ 5., 11., 17.],
        [11., 25., 39.],
        [17., 39., 61.]])
tensor([[ 5., 11., 17.],
        [11., 25., 39.],
        [17., 39., 61.]])


In [14]:
y1 = x_np @ x_np.T
y2 = x_np.T @ x_np

print(y1)
print(y2)

tensor([[ 5., 11., 17.],
        [11., 25., 39.],
        [17., 39., 61.]])
tensor([[35., 44.],
        [44., 56.]])


* 요소 곱(element-wise product)

In [15]:
z1 = x_np * x_np
z2 = x_np.mul(x_np)

print(z1)
print(z2)

tensor([[ 1.,  4.],
        [ 9., 16.],
        [25., 36.]])
tensor([[ 1.,  4.],
        [ 9., 16.],
        [25., 36.]])


* 단일 요소 텐서  
단일 요소 텐서로 부터 값을 뽑아서 사용하려면, .item()

In [16]:
agg = x_np.mean()
print(agg, type(agg))
print('='*30)

agg_item = agg.item() # .item()은 단일요소 텐서에서 사용 가능
print(agg_item, type(agg_item))

tensor(3.5000) <class 'torch.Tensor'>
3.5 <class 'float'>


# **2.Data Loader**

In [17]:
from torch.utils.data import DataLoader, TensorDataset

### (1) 데이터 준비


In [18]:
path = 'https://raw.githubusercontent.com/DA4BAM/dataset/master/advertising.csv'
data = pd.read_csv(path)
data = data[:10]  # 10건만 저장하기.
target = 'Sales'
x = data.drop(target, axis=1)
y = data.loc[:, target]

print(x.shape, y.shape)

(10, 3) (10,)


### (2) 텐서로 변환

* torch.tensor(numpy array, )
    * numpy array를 입력으로 받음

In [19]:
# pandas dataframe, series를 numpy array로 변환
# 현재 자료형 타입
print(type(x), type(y))
print('='*50)

# 넘파이로 바꾸기
x_np = x.values
y_np = y.values

print(type(x_np), type(y_np))
print(x_np.shape, y_np.shape)
print('='*50)

# y도 2차원 구조가 필요.
y_np = y_np.reshape(10, 1)  # 10행 1열 구조
print(x_np.shape, y_np.shape)

<class 'pandas.core.frame.DataFrame'> <class 'pandas.core.series.Series'>
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(10, 3) (10,)
(10, 3) (10, 1)


In [20]:
# 텐서로 변환
x_tensor = torch.tensor(x_np, dtype=torch.float32)
y_tensor = torch.tensor(y_np, dtype=torch.float32)

print(x_tensor.shape, y_tensor.shape)

torch.Size([10, 3]) torch.Size([10, 1])


* 텐서 내용 열어보기

In [21]:
x_tensor[:5]

tensor([[230.1000,  37.8000,  69.2000],
        [ 44.5000,  39.3000,  45.1000],
        [ 17.2000,  45.9000,  69.3000],
        [151.5000,  41.3000,  58.5000],
        [180.8000,  10.8000,  58.4000]])

In [22]:
y_tensor[:5]

tensor([[22.1000],
        [10.4000],
        [ 9.3000],
        [18.5000],
        [12.9000]])

### (3) Data Loader 준비
* Data Loader는 텐서 데이터셋을 입력으로 받음
    * 그래서, 텐서x, y를 하나의 **텐서 데이터셋**으로 합치기
    * 그 다음 DataLoader 생성

#### 1) 텐서 데이터셋

In [23]:
# TensorDataset 생성 : 텐서 데이터셋으로 합치기
dataset = TensorDataset(x_tensor, y_tensor)
dataset

In [24]:
# 첫 5개 행 조회
dataset[:5]

(tensor([[230.1000,  37.8000,  69.2000],
         [ 44.5000,  39.3000,  45.1000],
         [ 17.2000,  45.9000,  69.3000],
         [151.5000,  41.3000,  58.5000],
         [180.8000,  10.8000,  58.4000]]),
 tensor([[22.1000],
         [10.4000],
         [ 9.3000],
         [18.5000],
         [12.9000]]))

In [25]:
# 전체 조회
dataset.tensors

(tensor([[230.1000,  37.8000,  69.2000],
         [ 44.5000,  39.3000,  45.1000],
         [ 17.2000,  45.9000,  69.3000],
         [151.5000,  41.3000,  58.5000],
         [180.8000,  10.8000,  58.4000],
         [  8.7000,  48.9000,  75.0000],
         [ 57.5000,  32.8000,  23.5000],
         [120.2000,  19.6000,  11.6000],
         [  8.6000,   2.1000,   1.0000],
         [199.8000,   2.6000,  21.2000]]),
 tensor([[22.1000],
         [10.4000],
         [ 9.3000],
         [18.5000],
         [12.9000],
         [ 7.2000],
         [11.8000],
         [13.2000],
         [ 4.8000],
         [10.6000]]))

In [26]:
# x
dataset.tensors[0]

tensor([[230.1000,  37.8000,  69.2000],
        [ 44.5000,  39.3000,  45.1000],
        [ 17.2000,  45.9000,  69.3000],
        [151.5000,  41.3000,  58.5000],
        [180.8000,  10.8000,  58.4000],
        [  8.7000,  48.9000,  75.0000],
        [ 57.5000,  32.8000,  23.5000],
        [120.2000,  19.6000,  11.6000],
        [  8.6000,   2.1000,   1.0000],
        [199.8000,   2.6000,  21.2000]])

In [27]:
# y
dataset.tensors[1]

tensor([[22.1000],
        [10.4000],
        [ 9.3000],
        [18.5000],
        [12.9000],
        [ 7.2000],
        [11.8000],
        [13.2000],
        [ 4.8000],
        [10.6000]])

In [28]:
# 데이터 건수
len(dataset)

10

#### 2) 데이터로더

In [29]:
# 데이터 로더 생성
data_loader = DataLoader(dataset, batch_size = 3, shuffle = True)

* 데이터 로더로 부터 데이터 추출하기

In [30]:
# 데이터 로더로 부터 첫번째 배치 데이터 로딩
for (x_b, y_b) in data_loader:
    print(x_b)
    print(y_b)
    break

tensor([[ 8.7000, 48.9000, 75.0000],
        [44.5000, 39.3000, 45.1000],
        [57.5000, 32.8000, 23.5000]])
tensor([[ 7.2000],
        [10.4000],
        [11.8000]])


In [31]:
# 데이터 로더로부터 배치 단위로 데이터 로딩
for batch_id, (x_b, y_b) in enumerate(data_loader):
    print(f'batch_id : {batch_id}')
    print(x_b)
    print(y_b)
    print('='* 50)

batch_id : 0
tensor([[  8.7000,  48.9000,  75.0000],
        [  8.6000,   2.1000,   1.0000],
        [230.1000,  37.8000,  69.2000]])
tensor([[ 7.2000],
        [ 4.8000],
        [22.1000]])
batch_id : 1
tensor([[180.8000,  10.8000,  58.4000],
        [ 57.5000,  32.8000,  23.5000],
        [199.8000,   2.6000,  21.2000]])
tensor([[12.9000],
        [11.8000],
        [10.6000]])
batch_id : 2
tensor([[120.2000,  19.6000,  11.6000],
        [ 17.2000,  45.9000,  69.3000],
        [151.5000,  41.3000,  58.5000]])
tensor([[13.2000],
        [ 9.3000],
        [18.5000]])
batch_id : 3
tensor([[44.5000, 39.3000, 45.1000]])
tensor([[10.4000]])


In [32]:
data_loader.dataset

In [33]:
# dataloader의 데이터 셋
len(data_loader.dataset)

10

In [34]:
# 데이터로더의 배치 수
len(data_loader)

4

# **3.device**

* 텐서를 넘파이로 변환하려면,
    * device 할당된 텐서
        * cpu --> .numpy()
        * gpu --> .cpu().numpy()
    * device 할당 안된 텐서 : .numpy()

In [35]:
x_tensor.numpy()

array([[230.1,  37.8,  69.2],
       [ 44.5,  39.3,  45.1],
       [ 17.2,  45.9,  69.3],
       [151.5,  41.3,  58.5],
       [180.8,  10.8,  58.4],
       [  8.7,  48.9,  75. ],
       [ 57.5,  32.8,  23.5],
       [120.2,  19.6,  11.6],
       [  8.6,   2.1,   1. ],
       [199.8,   2.6,  21.2]], dtype=float32)